In [1]:
import src as dvc

/home/pqw/anaconda3/envs/virtual_cell/lib/python3.10/site-packages/cudf/utils/_ptxcompiler.py:64: UserWarning: Error getting driver and runtime versions:

stdout:



stderr:

Traceback (most recent call last):
  File "/home/pqw/anaconda3/envs/virtual_cell/lib/python3.10/site-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 277, in ensure_initialized
    self.cuInit(0)
  File "/home/pqw/anaconda3/envs/virtual_cell/lib/python3.10/site-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 327, in safe_cuda_api_call
    self._check_ctypes_error(fname, retcode)
  File "/home/pqw/anaconda3/envs/virtual_cell/lib/python3.10/site-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 395, in _check_ctypes_error
    raise CudaAPIError(retcode, msg)
numba.cuda.cudadrv.driver.CudaAPIError: [100] Call to cuInit results in CUDA_ERROR_NO_DEVICE

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<string>", line 4, in <module>
  Fi

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
sample_rep = "X_pca" # X_ae; X_state
control_key = "is_control"
condition_keys = "target_gene"
condition_rep_keys = "gene_embeddings"

In [5]:
filePath = './data/vcc_data/adata_Training.h5ad'
adata = sc.read_h5ad(filePath)

In [6]:
adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].copy()

In [7]:
adata.obs[control_key] = [(lambda x: True if x == "non-targeting" else False)(x) for x in adata.obs['target_gene']]

In [8]:
gene_list = adata[adata.obs[control_key]==False].obs['target_gene'].unique()
len(gene_list)

150

In [9]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
gene_list = adata[adata.obs[control_key]==False].obs['target_gene'].unique()

# train_gene = list(gene_list[2:]) + ["non-targeting"]
# test_gene = list(gene_list[:2])

# Starting from only one perturbation gene
train_gene = list(gene_list[0:1]) + ["non-targeting"]
test_gene = list(gene_list[2:4])

In [11]:
adata_train = adata[adata.obs["target_gene"].isin(train_gene)].copy()
adata_test = adata[adata.obs["target_gene"].isin(test_gene)].copy()

In [12]:
condition_rep_dict = pd.read_pickle("./data/vcc_data/vcc_data_target_genes_embedding.pkl")

In [13]:
adata_control, adata_treated, adata_test = dvc.pp.pp_with_cellflow(adata_train,
                adata_test = adata_test,
                    sample_rep = sample_rep, # X_ae; X_state
                    control_key = control_key,
                    condition_keys = condition_keys,
                    condition_rep_keys = condition_rep_keys,
                    condition_rep_dict = condition_rep_dict)

[1.0671057  0.5895     0.817349   ... 0.7491436  0.5844313  0.65452206]


/home/pqw/project/dvc/src/preprocessing/pp.py:70: ImplicitModificationWarning: Setting element `.obsm['gene_embeddings']` of view, initializing view as actual.
  adata_treated.obsm[condition_rep_keys] = convert_mixed_array_to_2d(temp)


In [14]:
# del adata, adata_train

In [15]:
in_out_dim = adata_control.obsm[sample_rep].shape[1]
hidden_dim_dyn = 256
n_hiddens_dyn = 6
condition_dim = adata_treated.obsm[condition_rep_keys].shape[1]
con_embedding_dim = 10
hidden_dim_con = 256
n_hiddens_con = 3
activation = 'relu'

model = dvc.tr.model.FNet(in_out_dim, hidden_dim_dyn, n_hiddens_dyn, 
                 condition_dim, con_embedding_dim, hidden_dim_con, n_hiddens_con, activation)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-5)

In [16]:
save_path = "./data/vcc_data/vcc_ot_precompute.pkl"
delta = 10.0
use_mini_batch_uot = False
group_number = 3

ot_results = dvc.tr.pre_compute_wfr_ot(adata_control, 
                       adata_treated, 
                       save_path, 
                       sample_rep=sample_rep, 
                       condition_keys=condition_keys,
                       delta = delta,
                       group_number = group_number,
                       use_mini_batch_uot = use_mini_batch_uot)

# ot_results = pd.read_pickle("./data/vcc_data/vcc_ot_precompute.pkl")

Computing UOT plans...:   0%|          | 0/1 [00:00<?, ?it/s]

Pre-computing WFR-OT for condition: CDCA2


Computing UOT plans...: 100%|██████████| 1/1 [00:02<00:00,  2.52s/it]

True


In [17]:
# print(ot_results["all_conditions"])
# # print(ot_results["control_obs_names"])
# # print(ot_results["treat_obs_names"][0])
# print(ot_results["gamma0_plans"][0].shape)
# print(ot_results["gamma1_plans"][0].shape)
# print(ot_results["delta"])

In [18]:
save_path = "./results/vcc/vcc_dvc_model.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_sample_rep = sample_rep + "_scaled"
# train_sample_rep = sample_rep

np.std(np.array(adata_control.obsm[sample_rep + "_scaled"]), axis=0)

array([0.9986546 , 1.0038998 , 0.997963  , 0.99963397, 1.0021858 ,
       1.0018119 , 0.99847656, 0.9989977 , 0.99785304, 1.0044018 ,
       0.99818224, 0.99743754, 1.0034539 , 0.99841183, 1.0034263 ,
       0.9979432 , 0.99913675, 1.0054152 , 1.0030578 , 0.9987771 ,
       1.0010717 , 1.0020759 , 1.0008651 , 1.0000134 , 1.0020641 ,
       1.0045719 , 0.99823093, 1.0001924 , 1.005804  , 1.0022484 ],
      dtype=float32)

In [19]:
dvc.tr.train.train_model(adata_control,
          adata_treated,
          ot_results,
          model.to(device),
          optimizer,
          reorder_ot_results = False,
          n_iterations=10000,
          batch_size_per_condition = 2560,
          batch_size_condition = 1,
          sample_rep = train_sample_rep, # X_ae; X_state
          condition_rep_keys = condition_rep_keys,
          device = device,
          save_path=save_path)

Begin flow and growth matching...:  25%|██▌       | 2514/10000 [01:45<05:13, 23.86epoch/s, loss=nan, vloss=nan, gloss=nan]


KeyboardInterrupt: 